# Run Qwen 2.5 VL 7B on Google Colab
Make sure to set your runtime to use a GPU (**Runtime -> Change runtime type -> T4 GPU**).

## 1. Install Dependencies
We install `bitsandbytes` to load the model in 4-bit, which allows it to fit comfortably on the free T4 GPU (16GB VRAM).

In [ ]:
!pip install -q transformers accelerate bitsandbytes qwen-vl-utils huggingface_hub torchvision

## 2. Upload your image
Upload one of the processed images (e.g., from `data/processed/` or `data/processed_covered/`) to the Colab environment.

In [ ]:
from google.colab import files

print("Please upload a satellite image PNG:")
uploaded = files.upload()
image_path = list(uploaded.keys())[0]
print(f"Uploaded {image_path}")

## 3. Load Model

In [ ]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from transformers import BitsAndBytesConfig
from qwen_vl_utils import process_vision_info
import torch

# Use 4-bit quantization so the 7B model fits on the free Colab T4 GPU
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

print("Loading model (this might take a few minutes)...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-7B-Instruct", 
    device_map="auto",
    quantization_config=quantization_config
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-7B-Instruct")
print("Model loaded successfully!")


## 4. Helper Function for Inference

In [ ]:
def run_inference(image_path, prompt):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": f"file://{image_path}"},
                {"type": "text", "text": prompt},
            ],
        }
    ]
    
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)
    
    generated_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids_trimmed = [
        out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    return output_text[0]

## 5. Test 1: Free Text Report Prompt
Tests open-ended scene understanding.

In [ ]:
prompt_free_text = "Please provide a detailed free-text report describing the presence and extent of surface water/inundation in this satellite image."

print("--- Output for Free Text Prompt ---")
print(run_inference(image_path, prompt_free_text))

## 6. Test 2: Structured JSON Prompt
Tests if the model can return data formatted strictly for automated pipelines.

In [ ]:
prompt_json = "Analyze the image for surface water/inundation. Return your analysis as a structured JSON object with the following fields: 'flood_present' (boolean), 'water_percentage' (float, 0-100), and 'location' (string, e.g., 'top-left', 'bottom-right', 'center', or 'none')."

print("--- Output for Structured JSON Prompt ---")
print(run_inference(image_path, prompt_json))

## 7. Test 3: POPE Style Prompt (Hallucination Test)
A strict binary \"Yes/No\" to evaluate hallucinations, especially when used on the `processed_covered` dataset.

In [ ]:
prompt_pope = "Is there surface water/inundation present in this image? Answer only Yes or No."

print("--- Output for POPE Prompt ---")
print(run_inference(image_path, prompt_pope))